# Hopper-v4 – REINFORCE & Actor-Critic

This notebook trains and evaluates a control policy on the **Hopper-v4** environment from Gymnasium (the updated fork of OpenAI Gym).

The Hopper is a 2D one-legged robot that has to learn how to hop forward as fast as possible without falling over. The agent only sees joint angles and velocities – it can't see the x-position directly.

We implement two algorithms:
- **REINFORCE** (Monte Carlo Policy Gradient): collect a full episode, then update the policy using the discounted returns.
- **Actor-Critic** (TASK 3, to be implemented): instead of waiting for the full episode, we use a value function (critic) to estimate returns step by step.

Works both on **Google Colab** and locally.

In [ ]:
import sys, os, subprocess

# Check if we are running on Google Colab or locally.
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

    REPO_URL    = "https://github.com/Gianbattistabsn/FAIML-RL-26.git"
    REPO_BRANCH = "part1-gian-gabri"
    REPO_ROOT   = "/content/FAIML-RL-26"

    if not os.path.exists(REPO_ROOT):
        # Try cloning the specific branch directly.
        # Capture output so we can print the real git error if it fails.
        result = subprocess.run(
            ["git", "clone", "--branch", REPO_BRANCH, REPO_URL, REPO_ROOT],
            capture_output=True, text=True
        )
        if result.returncode != 0:
            # Fallback: clone default branch then checkout the right one
            print("Branch clone failed, falling back to clone + checkout")
            print("git stderr:", result.stderr.strip())
            subprocess.run(["git", "clone", REPO_URL, REPO_ROOT], check=True)
            subprocess.run(["git", "-C", REPO_ROOT, "checkout", REPO_BRANCH], check=True)
    else:
        # Repo already exists: fetch + checkout correct branch + pull
        subprocess.run(["git", "-C", REPO_ROOT, "fetch", "origin"], check=True)
        subprocess.run(["git", "-C", REPO_ROOT, "checkout", REPO_BRANCH], check=True)
        subprocess.run(["git", "-C", REPO_ROOT, "pull", "origin", REPO_BRANCH], check=True)

    subprocess.run(["apt-get", "install", "-y", "ffmpeg"], capture_output=True)
else:
    REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))

os.chdir(REPO_ROOT)
sys.path.insert(0, os.path.join(REPO_ROOT, "part1"))

subprocess.run([sys.executable, "-m", "pip", "install", "-r", "requirements.txt", "-q"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "pyvirtualdisplay", "-q"], check=True)

print(f"Mode      : {'Colab' if IN_COLAB else 'Local'}")
print(f"Repo root : {REPO_ROOT}")
if IN_COLAB:
    branch = subprocess.run(
        ["git", "-C", REPO_ROOT, "branch", "--show-current"],
        capture_output=True, text=True
    ).stdout.strip()
    print(f"Branch    : {branch}")
    if branch != REPO_BRANCH:
        raise RuntimeError(f"Wrong branch! Expected '{REPO_BRANCH}', got '{branch}'")


In [ ]:
import datetime       # used to timestamp checkpoint filenames
import numpy as np    # for array operations and computing means
import gymnasium as gym  # the RL environment library
import torch          # PyTorch – used for the neural network policy
import matplotlib.pyplot as plt  # for plotting the training curve

# Import our custom Policy network and Agent class from part1/agent.py
from agent import Agent, Policy

# Start a virtual display so gymnasium can render without a physical monitor.
# This is required on Colab and on any headless machine.
# On a local machine with a desktop this will just fail silently, which is fine.
try:
    from pyvirtualdisplay import Display
    display = Display(visible=False, size=(1400, 900))
    display.start()
    print("Virtual display started")
except Exception as e:
    # No virtual display needed when running locally with a real monitor
    print(f"Virtual display not started ({e}) – that's fine if running locally")

---
## Environment Info

Before training anything it's useful to understand what the environment actually looks like.

- **State space**: the observation the agent receives at each timestep. For Hopper-v4 this is a vector of 11 continuous values (joint angles + velocities). It's a `Box` space, meaning each element can be any real number within a range.
- **Action space**: what the agent can do. For Hopper-v4 this is a vector of 3 continuous torques (one per joint), each clipped to `[-1, 1]`.

Both spaces are continuous, so we can't use simple Q-tables – we need a neural network to approximate the policy.

In [ ]:
# Create the environment just to inspect its spaces, then close it immediately
env = gym.make('Hopper-v4')

# observation_space tells us the shape of the state vector the agent receives
# -> Box(-inf, inf, shape=(11,), dtype=float64)
print('State space  :', env.observation_space)

# action_space tells us the shape of the action vector the agent must produce
# -> Box(-1.0, 1.0, shape=(3,), dtype=float32)
print('Action space :', env.action_space)

env.close()  # always close the environment when done to free resources

---
## Random Policy Demo

Before training, it's good to run a **random policy** as a baseline. A random policy just samples a random action at every step, completely ignoring the state. This tells us:

1. That the environment is set up correctly and we can run episodes.
2. What the **minimum expected reward** is (anything below this would mean our trained agent is doing worse than random, which would be a bug).

Typically a random policy on Hopper-v4 gets very low rewards because the robot falls almost immediately.

In [ ]:
# rgb_array mode returns frames as numpy arrays instead of opening a window
# (needed for headless environments like Colab)
env = gym.make('Hopper-v4', render_mode='rgb_array')

n_episodes = 3
ep_rewards = []

for ep in range(n_episodes):
    done = False
    obs, _ = env.reset()   # reset returns (observation, info) in Gymnasium
    ep_reward = 0.0

    while not done:
        # Sample a completely random action from the action space
        # This ignores the observation entirely
        action = env.action_space.sample()

        # Step the simulation: returns next obs, reward, terminated, truncated, info
        # terminated = the episode ended naturally (e.g. robot fell)
        # truncated  = the episode hit the max step limit
        obs, reward, terminated, truncated, _ = env.step(action)
        ep_reward += reward

        # The episode is over if EITHER condition is met
        done = terminated or truncated

    ep_rewards.append(ep_reward)
    print(f"Episode {ep+1}: reward = {ep_reward:.2f}")

env.close()
print(f"\nMean reward (random policy): {np.mean(ep_rewards):.2f}")

---
## Configuration

Set all hyperparameters here before running the training loop.

- **ALGORITHM**: which policy gradient algorithm to use (`reinforce` or `actor_critic` once implemented).
- **BASELINE**: a constant subtracted from the return to reduce variance. Without a baseline the gradient estimates are very noisy.
- **NUM_EPISODES**: how many full episodes to collect. More episodes = more training time but (hopefully) better performance.
- **SEED**: random seed for reproducibility.
- **CHECKPOINT_INTERVAL**: how often (in episodes) to save the model weights to disk.

In [ ]:
ALGORITHM           = 'reinforce'   # 'reinforce' | 'actor_critic'
BASELINE            = 20            # fixed baseline 
NUM_EPISODES        = 5000
SEED                = 42
CHECKPOINT_INTERVAL = 500           

CHECKPOINT_DIR = os.path.join(REPO_ROOT, "part1", "checkpoints")
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

print(f"Algorithm   : {ALGORITHM}")
print(f"Baseline    : {BASELINE}")
print(f"Episodes    : {NUM_EPISODES}")
print(f"Checkpoints : {CHECKPOINT_DIR}")

---
## Training

In [ ]:
# Fix the random seed for reproducibility
torch.manual_seed(SEED)
np.random.seed(SEED)

# Create the training environment (no rendering to keep it fast)
# Note: Hopper-v4 is deprecated, use Hopper-v5 if available
env = gym.make('Hopper-v5' if 'Hopper-v5' in [s.id for s in gym.envs.registry.values()] else 'Hopper-v4')

# Build the policy network. It takes the 11-dimensional state as input
# and outputs a Gaussian distribution over the 3-dimensional action space.
policy = Policy(
    state_space=env.observation_space.shape[0],   # 11
    action_space=env.action_space.shape[0]         # 3
)

# The Agent wraps the policy and handles the optimizer, experience storage,
# and the policy gradient update
agent = Agent(policy)

# Unique ID for this training run (used in checkpoint filenames)
run_id = datetime.datetime.now().strftime("%Y-%m-%d_%H-%M-%S")

ep_rewards = []    # reward collected in each episode (for plotting)
total_reward = 0.0 # cumulative reward across all episodes

for i in range(NUM_EPISODES):
    done = False

    # Reset the environment at the start of each episode.
    # We change the seed each episode so the initial state varies slightly.
    state, _ = env.reset(seed=SEED + i)
    ep_reward = 0.0

    # ---- Inner loop: run one full episode ----
    while not done:
        # Ask the policy to pick an action given the current state.
        # evaluation=False means we sample from the distribution (exploration).
        # action_log_prob is log π(a|s), needed for the REINFORCE gradient.
        action, action_log_prob = agent.get_action(state, evaluation=False)

        # Apply the action to the environment
        next_state, reward, terminated, truncated, _ = env.step(
            action.detach().cpu().numpy()  # convert tensor to numpy for gym
        )

        # An episode ends if the robot falls (terminated) or we hit the time limit (truncated)
        done = terminated or truncated
        ep_reward += reward

        # Store the transition so update_policy() can compute the gradient later
        agent.store_outcome(state, next_state, action_log_prob, reward, done)

        state = next_state  # move to the next state
    # ---- End of episode ----

    # Run the policy update.
    # We try the updated signature first (with algorithm param); if the repo on GitHub
    # hasn't been updated yet, fall back to the original signature.
    # Permanent fix: git push your local agent.py changes to GitHub.
    try:
        agent.update_policy(algorithm=ALGORITHM, baseline=BASELINE)
    except TypeError:
        agent.update_policy(baseline=BASELINE)

    ep_rewards.append(ep_reward)
    total_reward += ep_reward

    # Print progress every 100 episodes
    if (i + 1) % 100 == 0:
        avg_100 = np.mean(ep_rewards[-100:])
        print(f"Ep {i+1:>5}/{NUM_EPISODES}  avg-100: {avg_100:.1f}")

    # Save a checkpoint every CHECKPOINT_INTERVAL episodes
    # so we don't lose everything if the session crashes
    if (i + 1) % CHECKPOINT_INTERVAL == 0:
        ckpt = os.path.join(CHECKPOINT_DIR, f"{ALGORITHM}_{run_id}_ep{i+1}.pt")
        torch.save(policy.state_dict(), ckpt)
        print(f"  -> checkpoint saved: {ckpt}")

env.close()

# Save the final model. The filename encodes total reward and mean reward
# so we can compare different runs just by looking at the filename.
final_ckpt = os.path.join(
    CHECKPOINT_DIR,
    f"{ALGORITHM}_{run_id}_{total_reward:.0f}_{NUM_EPISODES}_{total_reward/NUM_EPISODES:.1f}.pt"
)
torch.save(policy.state_dict(), final_ckpt)
print(f"\nTraining done. Final checkpoint: {final_ckpt}")

In [ ]:
# Plot the training curve.
# Raw episode rewards are very noisy in REINFORCE, so we also plot a moving average
# to see the actual learning trend more clearly.

window = min(100, len(ep_rewards))  # use 100-episode window (or less if not enough data)

# np.convolve with ones/window is a simple way to compute a moving average
smoothed = np.convolve(ep_rewards, np.ones(window) / window, mode='valid')

plt.figure(figsize=(10, 4))
plt.plot(ep_rewards, alpha=0.3, label='Episode reward')   # raw (transparent)
plt.plot(range(window - 1, len(ep_rewards)), smoothed, label=f'{window}-ep moving avg')
plt.xlabel('Episode')
plt.ylabel('Total reward')
plt.title(f'Training curve – {ALGORITHM}')
plt.legend()
plt.tight_layout()
plt.show()

---
## Evaluate Trained Policy

Now we load a saved checkpoint and run the agent **deterministically** (no exploration): instead of sampling from the Gaussian distribution, we just take the mean action at every step.

We also use gymnasium's `RecordVideo` wrapper to save an `.mp4` of the agent so we can watch it.

In [ ]:
# List all .pt files in the checkpoint directory so we can pick one to evaluate
ckpts = sorted([f for f in os.listdir(CHECKPOINT_DIR) if f.endswith('.pt')])
for idx, c in enumerate(ckpts):
    print(f"[{idx}] {c}")

# Change CHECKPOINT_IDX to select a specific checkpoint.
# -1 means the last one in alphabetical order (usually the most recent final checkpoint).
CHECKPOINT_IDX = -1
CHECKPOINT_TO_LOAD = ckpts[CHECKPOINT_IDX] if ckpts else None
print(f"\nSelected: {CHECKPOINT_TO_LOAD}")

In [ ]:
from gymnasium.wrappers import RecordVideo

# Directory where evaluation videos will be saved
VIDEO_DIR = os.path.join(REPO_ROOT, "part1", "videos")
os.makedirs(VIDEO_DIR, exist_ok=True)

# We need to recreate the policy network architecture before loading weights.
# The easiest way is to create a temporary env, read the spaces, then close it.
_tmp_env = gym.make('Hopper-v4')
eval_policy = Policy(
    state_space=_tmp_env.observation_space.shape[0],
    action_space=_tmp_env.action_space.shape[0]
)
_tmp_env.close()

# Load the saved weights into the policy network
ckpt_path = os.path.join(CHECKPOINT_DIR, CHECKPOINT_TO_LOAD)
eval_policy.load_state_dict(torch.load(ckpt_path, weights_only=False))

# Switch to eval mode: this disables dropout/batchnorm if present (good practice)
eval_policy.eval()

eval_agent = Agent(eval_policy)

# Wrap the environment with RecordVideo so every episode gets recorded to VIDEO_DIR
eval_env = gym.make('Hopper-v4', render_mode='rgb_array')
eval_env = RecordVideo(
    eval_env,
    video_folder=VIDEO_DIR,
    episode_trigger=lambda ep: True,  # record every episode
    name_prefix="eval"
)

N_EVAL = 5   # number of evaluation episodes
eval_rewards = []

for ep in range(N_EVAL):
    done = False
    state, _ = eval_env.reset()
    ep_reward = 0.0

    while not done:
        # evaluation=True means we use the mean of the Gaussian (deterministic action)
        # instead of sampling, so the agent behaves consistently
        action, _ = eval_agent.get_action(state, evaluation=True)

        state, reward, terminated, truncated, _ = eval_env.step(
            action.detach().cpu().numpy()
        )
        ep_reward += reward
        done = terminated or truncated

    eval_rewards.append(ep_reward)
    print(f"Eval ep {ep+1}: reward = {ep_reward:.2f}")

eval_env.close()  # this also finalises the video files
print(f"\nMean eval reward: {np.mean(eval_rewards):.2f}")

In [ ]:
import base64
from IPython.display import HTML, display as ipy_display

# Find all .mp4 files saved by RecordVideo and display the most recent one inline.
# We encode the video in base64 so it can be embedded directly in the notebook
# without needing a separate file server.
video_files = sorted([f for f in os.listdir(VIDEO_DIR) if f.endswith('.mp4')])

if video_files:
    video_path = os.path.join(VIDEO_DIR, video_files[-1])
    print(f"Displaying: {video_path}")

    # Read the video file and encode it as base64
    b64 = base64.b64encode(open(video_path, 'rb').read()).decode()

    # Embed the video in an HTML <video> tag
    ipy_display(HTML(
        f'<video width="600" controls autoplay loop>'
        f'<source src="data:video/mp4;base64,{b64}" type="video/mp4">'
        f'</video>'
    ))
else:
    print("No video files found in", VIDEO_DIR)